In [ ]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [ ]:
SOURCE_DIR = Path("/workspaces/Arch_PC_Assistent/sources/raw_sources")
all_filepaths = list(SOURCE_DIR.rglob("*.txt"))
print(f"found files: {len(all_filepaths)}\n")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " "] # Die Reihenfolge, wie er versucht zu schneiden
)

all_chunks = []

print("Start Chunking-Prozess for all datas...\n")

for filepath in all_filepaths:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
        
    topic = filepath.parent.name
    filename = filepath.name

    chunks = text_splitter.create_documents([text])
    
    for chunk in chunks:
        chunk.metadata["topic"] = topic
        chunk.metadata["source_file"] = filename
        all_chunks.append(chunk)

print(f"Results: {len(all_chunks)} chunks was succesfully generated from {len(all_filepaths)} files.\n")

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
persist_directory = "/workspaces/Arch_PC_Assistent/embed_model"

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)


test_query = "How to install the GRUB bootloader?"
results = vectorstore.similarity_search(test_query, k=2)

for i, doc in enumerate(results):
    print(f"--- Hit {i+1} ---")
    print(f"Metadata: {doc.metadata}")
    print(f"Abstract: {doc.page_content[:250]}...\n")